# Quick DAXM: depth of the Laue spots of a wire scan, and wires calibration

From the wire scans (`ascan` / `fscan` of `zf` or `yf`) of your experiment, without reconstruction of the images:

| step | what | result (in `PROCESSED_DATA/<sample>/<dataset>/scanXXXX/daxm`) |
|---|---|---|
| 1 | choose a wire scan in the list of the experiment (`done`: results already there) | |
| 2 | settings file of the analysis (detector and wires calibrations found automatically) | `configs/daxm.yaml` |
| 3 | **quick daxm**: spots, intensity profiles, **sharp dips of the scattering signal** (the smooth fluorescence dips are removed by the local background) | `quickdaxm.h5` |
| 4 | (wire scan of a **reference crystal, Ge**) **wires calibration**: wires parameters from the dips of its Laue spots, then (option) refinement with the fluorescence (usual calibration) | `wirescalib/*.calib` |
| 5 | **depth of the spots** from the dips edges and the wires calibration; peak lists with a depth column, one per slice along the beam | `quickdaxm_spots.cor`, `depthslices/*.cor` |

**How to use it**: edit only the SETTINGS cell, then run the cells one after the other. Running the notebook again is safe: steps already done are reloaded (`REDO = True` to compute again). Images (`RAW_DATA`) are only read.

Wires calibration: notebook `1_wires_calibration` (crude from the Laue dips, fine with the fluorescence), or STEP 4 here for a Ge wire scan. Typical session: the calibration first, then the wire scans of the sample (steps 1, 2, 3, 5: the calibration of the last Ge scan before them is used, `CALIBFILE = 'auto'`).

Core functions: `LaueTools/Daxm/quickdaxm.py`, workflow: `LaueTools/daxm_workflow.py`. Units: `ybeam` in microns along the beam (> 0 deeper in the sample, 0 at the reference point of the wires calibration), `depth = ybeam * sin(sample tilt)`.

# SETTINGS

In [ ]:
# 1. your experiment
RAW_DATA_FOLDER = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA'
SCAN_FILTER = '<sample>_grid' # words of the dataset name ('-word': without it), e.g. 'Ge', 'daxm -grid'; '': all wire scans
SCAN_INDEX = '<dataset>_1' # None: choose the scan in the menu of STEP 1; or its index in the table or its dataset name
MODE = 'compute'           # 'compute': analyse (steps already done are reloaded); 'show': only show existing results

# 2. calibrations
DETFILE = 'auto'           # detector calibration .det file; 'auto': the last one acquired before the scan
CALIBFILE = 'auto'         # wires calibration (depth of the spots, STEP 5):
                           # 'auto': calibration made with a wire scan of the same sample folder (notebook
                           #   1_wires_calibration: refined, else crude), acquired last before the scan; else the Ge
                           #   wire scan of the folder: its approximate wires from its dips (computed now if needed)
                           # Ge wire scan: its dataset name (e.g. 'Zr4_Gedaxm_1') or index in the table (same rule)
                           # path of a .calib file
                           # wires by hand: [{'h': 0.78, 'p0': 3.58}, {'h': 0.79, 'p0': 3.93, 'f1': -0.45}, ...] (mm, deg)
                           # None: dips only, no depth. A Ge scan itself gets its wires from its own dips (STEP 4)
ENVELOPE_WIRES = False     # STEP 5 without wires calibration: approximate wires from the envelope of the dips of the
                           # scan itself (shallowest sources = depth 0 for each wire)
ENVELOPE_OUTLIERS = 0.02   # fraction of the dips of a wire allowed beyond its envelope line (outliers)
WIRES_CALIBRATION = 'auto' # STEP 4 (wires from the dips of a reference crystal): True, False, 'auto' (Ge in the dataset name)
NB_WIRES = None            # nb of wires (None: all the bands of dips found)
REFINE_FLUO = False        # STEP 4: refine the wires with the fluorescence (usual calibration, several minutes)
FLUO_WIRES = None          # STEP 4: wires refined with the fluorescence, e.g. [0, 1, 2] (the others keep their parameters
                           # from the dips); None: all the wires
EXCLUDED_WIRES = None      # wires whose dips give no depth (and not used by the reconstruction), e.g. [3] (a wire seen
                           # during a part of the scan only); None: the wires not in FLUO_WIRES
RADIUS_EACH_WIRE = False   # STEP 4: True: radius of each wire fitted to the dips (wires of different diameters,
                           # e.g. 50 and 80 microns); False: same radius for all the wires
FLUO_ELEMENT = 'Ge'        # element of the fluorescence of the reference crystal
CHECK_FLUO_PROFILES = False # STEP 4: check the intensity profiles used by the fluorescence refinement (plots, ~1 min)

# 3. quick daxm parameters (changes of the configuration file), e.g. {'ybeam_range': [-50, 150], 'spots_threshold': 200}
QUICKDAXM = {'skip_images': 40}  # changes of the quick daxm parameters. skip_images: first images of the scans not used
                           # (Zr5 grid: wire position wrong by ~8 µm in images 0-39 of the scans after the first one,
                           # see mismatch_vs_image())

# 4. computation and files
RESOURCE = 'magnifix' #'kernel'        # 'kernel' (this notebook), 'local' (background job), 'magnifix', 'nice-hpc7' ... (SLURM)
NB_CPUS = None             # None: all available cpus
LOCAL_FLUO_REFINE = None   # calibration of another scan (CALIBFILE, e.g. refined with notebook 1): refine only these
                           # wires parameters with the fluorescence of this scan, ['p0'] (depth zero of this place) or
                           # ['h', 'p0'] (seconds to a minute); None: calibration used as it is
LOCAL_FLUO_ELEMENT = 'Zr'  # LOCAL_FLUO_REFINE: fluorescent element of the sample (the layer of the calibration, e.g. Cr
                           # coating, is kept with its thickness)
DETECTOR_OFFSET = 'auto'   # microns along the beam: depth zero of the wires calibration relative to the source point of the
                           # detector calibration (2theta, chi of the spots); 'auto': thickness of the layer of the
                           # calibration (Zr fluorescence: Zr surface below the Cr coating whose top surface is at the
                           # Ge calibration point) + its p0 shift (other place of the sample); 0 for a Ge or envelope zero

# 3. indexation of the depth slices (STEP 5, peak lists)
INDEX_SLICES = False       # index from scratch all the .cor files of depthslices (raw UB matrix and material of each)
SLICE_MATERIALS = ['Cr', 'Zr']  # materials tried for each slice (the one indexing the most spots is kept)
SLICE_NB_GRAINS = 1        # max nb of grains per slice
INSTRUMENT_BLUR = 0.65     # microns (std along the beam, fluo_resolution() of notebook 1), removed from the grain thickness
REDO = False               # True: compute again the results of the selected scan
OUTPUT_ROOT = None         # None: PROCESSED_DATA of the experiment; or another folder for the results
CONFIG = None              # [batch] daxm configuration file of a scan: STEPS 1-2 skipped (RAW_DATA_FOLDER not used)

In [ ]:
# test on jupyter-slurm
# SCAN_FILTER = '<sample>'
# SCAN_INDEX = '<sample>_LaB6_in_daxm_2'
# MODE = 'compute'
# NVELOPE_WIRES = True
# OUTPUT_ROOT =  None # '<output folder>'
# CALIBFILE = None

# IMPORTS

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline

import LaueTools.daxm_workflow as DW
import LaueTools.Daxm.quickdaxm as QD
import LaueTools.generaltools as GT

# STEP 1: choose a wire scan

The table lists the wire scans of the experiment (read in the hdf5 files of the datasets, a few seconds):
- `points`: recorded wire positions, `images`: image files (1 file: all the images stacked in one file), `end`: end of the scan (not `SUCCESS`: interrupted)
- `done`: results already in `PROCESSED_DATA`: `config`, `quickdaxm`, `slices:<n>` (peak lists of the slices), `wires:dips`, `wires:fluo` (wires calibrations of STEP 4), `calib:<n>` (`.calib` files of the former notebooks `1_Pre_Calib_wire`, DAXM GUI)

Choose the scan in the menu below the table, then run the next cells.

In [ ]:
if CONFIG is None:
    wirescans = DW.list_wire_scans(RAW_DATA_FOLDER, contains=SCAN_FILTER, output_root=OUTPUT_ROOT)
    DW.show_wire_scans(wirescans)

In [ ]:
if CONFIG is None:
    selector = DW.WireScanSelector(wirescans, SCAN_INDEX)

# STEP 2: settings file of the analysis

`configs/daxm.yaml` in the folder of the results of the scan: written at the first run with the detector and wires calibrations (`DETFILE`, `CALIBFILE`) and `QUICKDAXM`; afterwards the file is kept with your edits, only these settings are updated.

In [ ]:
if CONFIG is None:
    index = selector.index
    row = wirescans.loc[index]
    print(f"scan {index}: {row['dataset']}  {row['command']}  ({row['end']})")
    if row['end'] != 'SUCCESS':
        GT.printyellow(f"this scan ended with {row['end']}: some images may be missing")
    if MODE == 'show':
        cfg = DW.load_config(Path(row['workdir']) / 'configs' / 'daxm.yaml')
    else:
        cfg = DW.config_for_scan(wirescans, index, detfile=DETFILE, calibfile=CALIBFILE, nb_cpus=NB_CPUS, **QUICKDAXM)
    dataset = row['dataset']
else:
    cfg = DW.load_config(CONFIG)
    dataset = cfg['name']
reference = DW.is_reference_scan(dataset) if WIRES_CALIBRATION == 'auto' else bool(WIRES_CALIBRATION)
cfg['sample']['detector_offset'] = DETECTOR_OFFSET
DW.check_config(cfg)
print('results folder:', cfg['output']['folder'])
print('wires calibration:', cfg['wires']['calibfile'], '| reference crystal (STEP 4):', reference)

**Local refinement of the wires calibration** (`LOCAL_FLUO_REFINE`): the wires parameters describe the wires relative to the beam and the detector; at another place of the sample only the depth zero changes (surface height). Only p0 (or h, p0) of the calibration are refined with the fluorescence of this scan (`DW.refine_wires_local()`), the result is used for the depth of the spots.

In [ ]:
if LOCAL_FLUO_REFINE and cfg['wires']['calibfile'] and MODE != 'show' and not reference:
    cfg['wirescalib']['fluo_element'] = LOCAL_FLUO_ELEMENT
    local = DW.refine_wires_local(cfg, cfg['wires']['calibfile'], var=LOCAL_FLUO_REFINE, replace_bad_points=True,
                                  nb_cpus=NB_CPUS)
    cfg['wires']['calibfile'] = local[-1]['calibfile']
    print('wires calibration refined at this place:', cfg['wires']['calibfile'])

# STEP 3: quick daxm (spots, intensity profiles, dips)

Results: `<results folder>/quickdaxm.h5`, reloaded at the next run. From a terminal: `python -m LaueTools.daxm_workflow quickdaxm <results folder>/configs/daxm.yaml --resource magnifix`

In [ ]:
if MODE == 'show':
    results = DW.load_quickdaxm(cfg)
else:  # 'compute'
    results = DW.run_quickdaxm(cfg, resource=RESOURCE, nb_cpus=NB_CPUS, redo=REDO)
spots, dips = results['spots'], results['dips']
print(f"{len(spots['X'])} spots, {len(dips['spot'])} dips, {int(np.sum(spots['nb_dips'] > 0))} spots with a depth")

## check the dips

Blue: spot minus local background (the analysed profile). Orange: local background (mostly fluorescence, with its smooth dips). Red and green lines: entry and exit edges of the dips found. Grey bands (with a wires calibration): images where each wire would mask a source at `ybeam = 0`.

If good dips are missed or bad ones kept, change `low_fraction`, `min_edge_fraction`, `min_snr`, `boxsize` in the configuration file and analyse again the collected profiles with the option cell below (a second, no image read).

In [ ]:
SPOTS = list(range(6))   # indices of spots (sorted by decreasing intensity)

plt.close('all')   # figures of the previous cells closed (not shown again)
fig, axes = plt.subplots(len(SPOTS) // 2, 2, figsize=(14, 3.2 * (len(SPOTS) // 2)))
for ax, spot in zip(axes.ravel(), SPOTS):
    QD.plot_profile(results, spot, ax)
fig.tight_layout()
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

In [ ]:
# OPTION: analyse again the collected profiles with the current configuration file (fast)
if 0:
    cfg = DW.load_config(cfg['configfile'])
    results['params'] = {key: cfg['quickdaxm'][key] for key in QD.DEFAULT_PARAMS}
    results['dips'], results['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(results), results['params'])
    QD.save_results(results, DW.output_files(cfg)['quickdaxm'])
    spots, dips = results['spots'], results['dips']

# STEP 4: wires calibration (wire scan of a reference crystal, Ge)

Run only for a reference crystal (`WIRES_CALIBRATION`). The Laue spots of Ge come from the surface (within a few microns): their dips give the wires parameters.
1. **approximate wires from the dips**: the dips of each wire lie on a band of the plot *image index of dip* vs *pixel Y* (the lines `WP` of the former notebooks); a least squares fit of `h`, `p0`, `f1` per wire and the radius `R` to the dips edges. Wire 0: largest pixel Y of its shadow (closest to the sample surface). The wire whose shadow is seen only during a part of the scan has the fewest dips: check it
2. check: the plot of the dips with the lines of these wires, and Laue images with the predicted shadows
3. (option `CHECK_FLUO_PROFILES`) check of the intensity profiles that the refinement will fit (dip depth, noise, complete dip, spikes, offset from the predicted dip)
4. (option `REFINE_FLUO`) **refinement with the fluorescence** (`FLUO_WIRES`: only some wires, e.g. `[0, 1, 2]`; the other wires keep their parameters from the dips and are excluded from the depth analysis and the reconstruction, `EXCLUDED_WIRES`, key `excluded_wires` of the `.calib` file: their shadows stay known) (CalibFluo, stages of `calib.py`), starting from these wires: a few minutes per stage

The resulting `.calib` file is found automatically (`CALIBFILE = 'auto'`) for the wire scans of the sample acquired after this scan.

In [ ]:
print('reference',reference)

if reference and MODE != 'show':
    cfg['wirescalib']['var'] = ['h', 'p0', 'f1', 'Rw' if RADIUS_EACH_WIRE else 'R']
    wires_fit = DW.approximate_wires(cfg, results)
    calibfile = wires_fit['calibfile']
    excluded = EXCLUDED_WIRES if EXCLUDED_WIRES is not None else (
        None if FLUO_WIRES is None else sorted(set(range(len(wires_fit['wires']))) - set(FLUO_WIRES)))
    QD.set_excluded_wires(calibfile, excluded)
    if excluded:
        print(f'wires excluded from the depth analysis and the reconstruction: {excluded}')
elif reference:
    calibfile = DW.calib_files(cfg)['dips']

In [ ]:
# dips (colored by the wire they are assigned to) and lines of the approximate wires for sources at ybeam = 0
plt.close('all')   # figures of the previous cells closed (not shown again)
if reference:
    check = dict(results, calibfile=calibfile)
    check['dips'], check['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(check), results['params'])
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    QD.plot_abacus(check, axes[0])
    QD.plot_histogram(check, axes[1], step=1.)
    axes[1].set_title('Laue spots of the reference crystal: expected close to ybeam 0')
    fig.tight_layout()
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

In [ ]:
IMAGES = None   # image indices of the Laue patterns shown with the predicted shadows (None: 3 images of the scan)

plt.close('all')   # figures of the previous cells closed (not shown again)
if reference:
    geometry = QD.WireGeometry.from_files(calibfile, cfg['detector']['detfile'], results['wire_positions'])
    nbimages = len(results['wire_positions'])
    images = IMAGES or [nbimages // 6, nbimages // 2, 5 * nbimages // 6]
    fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 6))
    for ax, imageindex in zip(np.atleast_1d(axes), images):
        QD.plot_image_shadows(cfg['scan'], geometry, imageindex, ax=ax)
    fig.tight_layout()
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

In [ ]:
# OPTION: check of the intensity profiles used by the fluorescence refinement (same points, same window of wire
# positions, predicted from the wires above). Usable profile (green): deep enough dip (contrast), above the noise (snr),
# complete dip in the window, no spike (diffraction spot or hot pixel), dip center close to the predicted one.
# Many red profiles of a wire: its starting parameters or its points need attention before REFINE_FLUO
CHECK_DIMS = (3, 3)   # grid of points per wire (stage 2 of the refinement: (3, 3); stage 3: (6, 6), with data_span=3.5)

plt.close('all')
if reference and CHECK_FLUO_PROFILES:
    cfg['wirescalib']['fluo_element'] = FLUO_ELEMENT
    profiles_table = DW.check_fluo_profiles(cfg, calibfile, dims=CHECK_DIMS)
    display(profiles_table)
plt.show()

In [ ]:
# OPTION: refinement with the fluorescence (usual wires calibration), starting from the approximate wires
REFINE_FLUO = True

print('REFINE_FLUO', REFINE_FLUO)

if reference and REFINE_FLUO and MODE != 'show':
    cfg['wirescalib']['fluo_element'] = FLUO_ELEMENT
    stages = DW.refine_wires_fluo(cfg, init_calib=calibfile, wires=FLUO_WIRES, excluded_wires=EXCLUDED_WIRES)
    calibfile = stages[-1]['calibfile']
    print('wires calibration refined with the fluorescence:', calibfile)

In [ ]:
# use the wires calibration of this reference scan for the depth of its own spots (STEP 5)
if reference:
    cfg = DW.new_config(cfg['configfile'], calibfile=calibfile)
    results['calibfile'] = calibfile
    results['dips'], results['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(results), results['params'])
    QD.save_results(results, DW.output_files(cfg)['quickdaxm'])
    spots, dips = results['spots'], results['dips']

# STEP 5: depth of the Laue spots

Needs a wires calibration (`CALIBFILE`, or STEP 4 for a reference scan).

## dips and wires calibration

Image index of the dips vs pixel Y of the spots, colored by their position along the beam. Lines: shadow of each wire calculated from the `.calib` file for sources at `ybeam = 0` (solid) and 50 µm (dashed); spots of a grain lie on a line parallel to them. `wp_parameters()`: equivalent `[a, b]` of the lines `WP` of the former quick daxm notebooks (`image index = a / 450 * Y + b`).

## (option) wires from the envelope of the dips, without wires calibration

`ENVELOPE_WIRES = True` and no wires calibration (`CALIBFILE = None`, or none found): all the sources are below the surface, so in the plot of the dips (image index vs pixel Y) the dips of each wire lie on one side of the line of the sources at depth 0. For each band of dips (one per wire), this line is its envelope: lines of a common slope with only `ENVELOPE_OUTLIERS` of the dips beyond. They give approximate wires (h, p0 per wire, radius from the widths of the dips), saved in `wirescalib/wires_from_envelope.calib`.

Right plot: vertical distance of each dip to the line of its wire (image index of the dip minus image index of the line at the same pixel Y; about microns along the beam for a wire step of 1 µm): a sharp start at 0 is expected, it is about the distribution of the positions of the sources along the beam. The depth zero of each wire is its shallowest source: the surface if grains of the surface scatter in its region of the detector. Validated on a Ge scan (lines within 7 µm in p0, 15 µm in h of the wires fitted to its dips). A wire with few dips (e.g. seen during a part of the scan only) gets a poor line: `EXCLUDED_WIRES`.

In [ ]:
plt.close('all')
if ENVELOPE_WIRES and QD.geometry_of_results(results) is None and MODE != 'show':
    envelope = QD.wires_from_envelope(results, nb_wires=NB_WIRES, outliers=ENVELOPE_OUTLIERS)
    QD.plot_envelope(results, envelope)
    folder = DW.calib_files(cfg)['folder']
    Path(folder).mkdir(parents=True, exist_ok=True)
    calibfile = QD.save_calib(envelope, 'wires_from_envelope.calib', directory=folder, excluded_wires=EXCLUDED_WIRES)
    print('approximate wires from the envelope of the dips:', calibfile)
    cfg = DW.new_config(cfg['configfile'], calibfile=calibfile)
    results['calibfile'] = calibfile
    results['dips'], results['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(results), results['params'])
    QD.save_results(results, DW.output_files(cfg)['quickdaxm'])
    spots, dips = results['spots'], results['dips']
plt.show()

In [ ]:
geometry = QD.geometry_of_results(results)
plt.close('all')   # figures of the previous cells closed (not shown again)
if geometry is None:
    GT.printyellow('no wires calibration (CALIBFILE): dips only, no depth; the next cells need it')
else:
    QD.plot_abacus(results, ybeams=(0, 50))
    print('equivalent WP parameters [a, b] of each wire:')
    for iw, ab in enumerate(geometry.wp_parameters()):
        print(f'  wire {iw}: [{ab[0]:.0f}, {ab[1]:.0f}]')
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

## position of the spots along the beam

In [ ]:
plt.close('all')   # figures of the previous cells closed (not shown again)
if geometry is not None:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={'width_ratios': [1.3, 1]})
    QD.plot_histogram(results, axes[0], step=2.)
    QD.plot_spots(results, axes[1])
    fig.tight_layout()
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

**Quality.** Each complete dip gives `ybeam` twice (entry and exit edges): `mismatch` is their difference. A spot masked by two wires gives two dips: `spread` is the difference of their `ybeam`. Both check the wires calibration (a few microns expected). `extent`: size of the source along the beam from the blur of the edges.

In [ ]:
plt.close('all')   # figures of the previous cells closed (not shown again)
if geometry is not None:
    complete = np.isfinite(dips['mismatch'])
    twowires = spots['nb_dips'] > 1
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
    axes[0].hist(dips['mismatch'][complete], bins=np.arange(-10, 10.5, 0.5))
    axes[0].set_xlabel('mismatch entry - exit (µm)')
    axes[1].hist(spots['spread'][twowires], bins=np.arange(0, 20.5, 0.5))
    axes[1].set_xlabel('spread between 2 wires (µm)')
    axes[2].hist(spots['extent'][np.isfinite(spots['extent'])], bins=np.arange(0, 20.5, 0.5))
    axes[2].set_xlabel('extent of the source (µm, std)')
    fig.tight_layout()
    print(f"mismatch: median {np.median(dips['mismatch'][complete]):.1f} µm;  "
          f"spread between 2 wires: median {np.median(spots['spread'][twowires]):.1f} µm ({twowires.sum()} spots)")
    # mismatch as a function of the image index of the edges: a step reveals a wrong wire position in a part of the
    # scan (e.g. first ~40 images of the scans of a grid after the return of the wire): then QUICKDAXM skip_images
    display(QD.mismatch_vs_image(results, geometry).round(2).T)
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)

## peak lists

- `quickdaxm_spots.cor`: all spots with a position, with columns `ybeam`, `depth`, `extent` (µm), `wire`, `nb_dips` (2theta and chi for the source of each spot)
- `depthslices/<name>_0000.cor ...`: one file per slice along the beam (`slice_step`, `slice_thickness`, `slice_min_spots` of the configuration file), 2theta and chi for the center of the slice; `depthslices/slices.yaml` lists them. The series can be indexed like a line scan (`notebooks/indexation`, `cor_folder` = `depthslices`)

In [ ]:
if geometry is not None and MODE != 'show':
    files = DW.write_quickdaxm_files(cfg, results)
    print([(s['index'], s['ybeam'], s['nb_spots']) for s in files['slices'] if s['file']])

## browser of the peak lists: Laue pattern in gnomonic projection

Spots of each peak list on the detector (left) and in gnomonic projection (right): the spots of a grain lie on lines (zone axes), a check that a slice along the beam contains whole Laue patterns before its indexation. Slider: file (0: all the spots, `quickdaxm_spots.cor`, then the slices along the beam); `nb spots`: most intense spots shown (0: all); `all spots in grey`: all the spots of the scan as background. Colors: `ybeam` (same scale for all the files), size: intensity.

In [ ]:
BROWSE_PEAKLISTS = True
NB_SPOTS_SHOWN = 200   # most intense spots of a file shown (None: all)

plt.close('all')
if geometry is not None and BROWSE_PEAKLISTS:
    files_cor = DW.output_files(cfg)
    peaklists = [files_cor['quickdaxm_cor']] + sorted(str(p) for p in Path(files_cor['quickdaxm_slices']).glob('*.cor'))
    peaklists = [f for f in peaklists if Path(f).exists()]
    if peaklists:
        browser = QD.PeakListBrowser(peaklists, nbmax=NB_SPOTS_SHOWN, background_file=peaklists[0])
    else:
        GT.printyellow('no peak list: run the cell above (peak lists)')

## indexation of the depth slices (option)

`INDEX_SLICES = True`: each `.cor` file of `depthslices` is indexed from scratch, trying each material of `SLICE_MATERIALS` (LaueTools indexation, sequence `best_of`: the material indexing the most spots is kept; configuration `configs/indexing_slices.yaml`, `.fit` files in `indexing_slices/fitfiles_<material>`). The table gives for each grain found: material, raw UB matrix (LaueTools frame), nb of indexed spots, and its position along the beam from the dips of its spots:

- `ybeam`: median of the half height positions of its dips (middle of the grain); `ybeam_min`, `ybeam_max`: medians of the onsets (shallowest sources) and ends (deepest sources) of the edges of its dips
- `thickness_beam`: thickness of the grain along the beam, from the width of the edges of the dips with the blur of the instrument (`INSTRUMENT_BLUR`) removed (uniform grain); `thickness_beam_spread`: its spread over the dips; `depth*`, `thickness_depth`: the same normal to the surface (x sin 40°)
- all relative to the depth zero of the wires calibration; `ybeam_detector`: relative to the source point of the detector calibration (`DETECTOR_OFFSET`), the source position to use to refine the UB matrix

The UB matrices are also written in `indexing_slices/slices_ub.yaml` (section `ub_matrices`, names `<material>_s<slice>_g<grain>`), to be refined with the proper material and the mean depth of the grain. The strain refined from all the spots of a grain is its average over its thickness along the beam.

In [ ]:
INDEX_SLICES

In [ ]:
INDEX_SLICES = True

In [ ]:
slices_ub = None
if INDEX_SLICES and geometry is not None and MODE != 'show':
    DW.index_slices(cfg, materials=SLICE_MATERIALS, nb_grains=SLICE_NB_GRAINS, nb_cpus=NB_CPUS)
if geometry is not None and (Path(cfg['output']['folder']) / 'indexing_slices').exists():
    slices_ub = DW.slices_ub_table(cfg, results, sigma=INSTRUMENT_BLUR)
    if len(slices_ub):
        display(slices_ub[['slice', 'slice_ybeam', 'material', 'grain', 'indexed', 'pixdev', 'nb_dips', 'ybeam', 'ybeam_min',
                           'ybeam_max', 'thickness_beam', 'thickness_beam_spread', 'depth', 'thickness_depth',
                           'ybeam_detector']].round(2))
        print('UB matrices:', Path(cfg['output']['folder']) / 'indexing_slices' / 'slices_ub.yaml')
    else:
        GT.printyellow('no grain found in the slices')

In [ ]:
slices_ub.UB

### visualisation of the indexation of the slices

Left: the grains along the beam, one line per grain (the same grain indexed in several slices, misorientation < `SAME_GRAIN_ANGLE` with the symmetry of its material, is one line): bar from the onset to the end of the edges of the dips of its spots (blurred limits; label: thickness with the blur removed), points: the slices where it was indexed (size ~ nb of indexed spots). Solid line: depth zero of the wires calibration; dashed: source point of the detector calibration. Right: nb of indexed spots of each slice per material, and pixel deviation (points).

In [ ]:
SAME_GRAIN_ANGLE = 1.0   # deg: grains of the slices with a smaller misorientation are the same grain

plt.close('all')
if slices_ub is not None and len(slices_ub):
    offset = QD.detector_offset_um(dict(results, detector_offset=cfg['sample'].get('detector_offset', 'auto')))
    slices_ub = DW.plot_slices_grains(slices_ub, threshold=SAME_GRAIN_ANGLE, detector_offset=offset,
                                      sample_tilt=cfg['sample']['tilt'])
    display(slices_ub.groupby('grain_id').agg(material=('material', 'first'), slices=('slice', list),
                                              indexed=('indexed', 'sum'), ybeam=('ybeam', 'median'),
                                              thickness_beam=('thickness_beam', 'median'),
                                              depth=('depth', 'median')).sort_values('ybeam').round(2))
plt.show()

### refinement of each grain at its own depth (option)

`REFINE_GRAINS = True`: for each grain of the slices (same grain in several slices: one grain), the spots of the scan with `ybeam` within the onset - end of its dips (± `GRAIN_MARGIN` µm) are written in `grains/grain_<n>.cor` with 2theta and chi computed for a source at the median `ybeam` of the grain (parallax of its own position along the beam, + `DETECTOR_OFFSET`), and its UB matrix (from the slice where it indexes the most spots) is refined with its material: orientation and deviatoric strain averaged over the thickness of the grain along the beam. `.fit` files in `grains/fitfiles`, each with a DAXM block at the end (`#DAXM key: value`: grain id, slices, source position, ybeam, ybeam_min, ybeam_max, thickness_beam, depth, depth_min, depth_max, thickness_depth, calibration files; `DW.read_daxm_block(fitfile)`); table also in `grains/grains.json`.

In [ ]:
REFINE_GRAINS = True # False
GRAIN_MARGIN = 2.     # microns along the beam added to the onset - end range of a grain to select its spots

grains_table = None
if REFINE_GRAINS and slices_ub is not None and len(slices_ub) and MODE != 'show':
    grains_table = DW.refine_slice_grains(cfg, slices_ub, results, threshold=SAME_GRAIN_ANGLE, margin=GRAIN_MARGIN,
                                          sigma=INSTRUMENT_BLUR)
elif (Path(cfg['output']['folder']) / 'grains' / 'grains.json').exists():
    import pandas as pd
    grains_table = pd.read_json(Path(cfg['output']['folder']) / 'grains' / 'grains.json')
if grains_table is None:
    if slices_ub is None or not len(slices_ub):
        GT.printyellow('no grain of the slices: run the indexation of the slices first (INDEX_SLICES = True)')
    else:
        GT.printyellow('grains not refined yet: set REFINE_GRAINS = True in this cell and run it again '
                       f'(results in {Path(cfg["output"]["folder"]) / "grains"})')
elif not len(grains_table):
    GT.printyellow('no grain refined')
else:
    columns = ['grain_id', 'material', 'slices', 'indexed', 'pixdev', 'source_ybeam', 'ybeam', 'ybeam_min', 'ybeam_max',
               'thickness_beam', 'depth', 'depth_min', 'depth_max', 'eps_sample_xx', 'eps_sample_yy', 'eps_sample_zz',
               'eps_sample_yz', 'eps_sample_xz', 'eps_sample_xy', 'a', 'b', 'c', 'alpha', 'beta', 'gamma']
    display(grains_table[[c for c in columns if c in grains_table]].round(3))

### unit cell of each grain in the sample frame

Browser (slider: grain) of the unit cell of the grains (refined grains if computed, else the UB matrices of the slices): 3D view and projections seen from above the sample (Zs) and from the sides, in the LaueTools sample frame (Zs normal to the surface, Xs in the plane of the beam and Zs, Ys horizontal); arrows a (red), b (green), c (blue), incoming beam (orange). Title: crystal directions along Zs, Xs and the beam (and the angle of the c axis to the normal for hexagonal cells). `LaueTools/unitcellview.py`: also `UV.plot_cell_views(UB, lattice parameters)` for one matrix, `UV.grains_from_fitfiles(files)` for any .fit files.

In [ ]:
import LaueTools.unitcellview as UV

plt.close('all')
cells = []
if grains_table is not None and 'UB' in grains_table:
    cells = [{'UB': row['UB'], 'material': row['material'],
              'label': f"{row['grain_id']}: ybeam {row['ybeam']:.1f} µm (depth {row['depth']:.1f} µm), "
                       f"{row['thickness_beam']:.1f} µm thick along the beam"}
             for _, row in grains_table.iterrows() if isinstance(row.get('UB'), list)]
elif slices_ub is not None and len(slices_ub):
    cells = [{'UB': row['UB'], 'material': row['material'],
              'label': f"slice {row['slice']} {row['material']} grain {row['grain']}: ybeam {row['ybeam']:.1f} µm"}
             for _, row in slices_ub.iterrows()]
if cells:
    cell_browser = UV.CellBrowser(cells, sample_tilt=cfg['sample']['tilt'])

## spots at a given position along the beam

Spots likely to belong to the grain(s) at `YBEAM` ± `TOLERANCE`, strongest first.

In [ ]:
YBEAM, TOLERANCE = 20., 3.   # microns

plt.close('all')   # figures of the previous cells closed (not shown again)
if geometry is not None:
    indices = QD.select_spots(results, (YBEAM - TOLERANCE, YBEAM + TOLERANCE))
    print(f'{len(indices)} spots at ybeam = {YBEAM} +/- {TOLERANCE} µm (depth {geometry.depth(YBEAM):.1f} µm)')
    print('    X       Y      intensity  ybeam  wire')
    for k in indices[:20]:
        print(f"{spots['X'][k]:7.0f} {spots['Y'][k]:7.0f} {spots['intensity'][k]:10.0f} {spots['ybeam'][k]:6.1f} {int(spots['wire'][k]):4d}")
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.plot(spots['X'], spots['Y'], '.', color='0.8', ms=3)
    ax.scatter(spots['X'][indices], spots['Y'][indices], s=20, c='r')
    ax.set_xlim(0, 2 * geometry.ccd_params[1]); ax.set_ylim(2 * geometry.ccd_params[2], 0); ax.set_aspect('equal')
    ax.set_xlabel('X (pixel)'); ax.set_ylabel('Y (pixel)')
    # QD.write_cor(results, 'grain_ybeam20', cfg['output']['folder'], indices, ybeam_for_angles=YBEAM)   # .cor of these spots
plt.show()   # displays the figure(s) of the cell (needed with %matplotlib widget on some servers)